In [3]:
!python -m pip install paddlepaddle==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/                    
!python -m pip install "paddleocr[all]"                           
!pip install torch transformers                                      
!python -m pip install label-studio 


Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cpu/


In [1]:
import json

from paddleocr import PaddleOCR

import os
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"


ocr = PaddleOCR(
    use_doc_orientation_classify=False, 
    use_doc_unwarping=False, 
    use_textline_orientation=False) # text detection + text recognition



image_path = "./training data/1.jpg"


result = ocr.predict(image_path) # predict the image
for res in result:
    res.print() # print the result of each image
    # res.save_to_img("output")
    res.save_to_json("output/paddle_ocr_result.json")



D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\PP-OCRv5_server_det`.
Cre

In [28]:
import json

def convert_ocr_to_ls(ocr_json, image_url, img_w, img_h):

    results = []

    polys = ocr_json["rec_polys"]
    texts = ocr_json["rec_texts"]

    for i, poly in enumerate(polys):

        xs = [p[0] for p in poly]
        ys = [p[1] for p in poly]

        x_min = min(xs)
        y_min = min(ys)

        x_max = max(xs)
        y_max = max(ys)

        w = x_max - x_min
        h = y_max - y_min

        results.append({
            "id": f"box_{i}",
            "type": "rectangle",
            "from_name": "bbox",
            "to_name": "image",
            "original_width": img_w,
            "original_height": img_h,
            "image_rotation": 0,
            "value": {
                "x": x_min/img_w*100,
                "y": y_min/img_h*100,
                "width": w/img_w*100,
                "height": h/img_h*100,
                "rotation": 0
            }
        })
        
        # OCR Text
        results.append({
            "id": f"box_{i}",
            "type": "textarea",
            "from_name": "transcription",
            "to_name": "image",
            "value": {
                "text": [texts[i]]
            }
        })
        
    return {
        "data": {
            "ocr": image_url
        },
        "predictions":[
            {
                "model_version":"paddleocr",
                "result": results
            }
        ]
    }

data=convert_ocr_to_ls(res, "http://localhost:8000/output/1.jpg", 2232, 3300)
print(data)
                       
# with open("./output/paddle_ocr_result.json", "r") as f:
#     data = json.load(f)
#     print(data.get('ocr'))
with open("output/labelstudio_tasks.json", "w") as f:
    json.dump(data, f, indent=4)


{'data': {'ocr': 'http://localhost:8000/output/1.jpg'}, 'predictions': [{'model_version': 'paddleocr', 'result': [{'id': 'box_0', 'type': 'rectangle', 'from_name': 'bbox', 'to_name': 'image', 'original_width': 2232, 'original_height': 3300, 'image_rotation': 0, 'value': {'x': np.float64(63.799283154121866), 'y': np.float64(0.27272727272727276), 'width': np.float64(9.722222222222223), 'height': np.float64(2.4242424242424243), 'rotation': 0}}, {'id': 'box_0', 'type': 'textarea', 'from_name': 'transcription', 'to_name': 'image', 'value': {'text': ['P_45']}}, {'id': 'box_1', 'type': 'rectangle', 'from_name': 'bbox', 'to_name': 'image', 'original_width': 2232, 'original_height': 3300, 'image_rotation': 0, 'value': {'x': np.float64(44.31003584229391), 'y': np.float64(1.7878787878787878), 'width': np.float64(10.170250896057349), 'height': np.float64(1.4242424242424243), 'rotation': 0}}, {'id': 'box_1', 'type': 'textarea', 'from_name': 'transcription', 'to_name': 'image', 'value': {'text': ['T